# 08 — Tái lập benchmark Feedzai trên Base

Chạy **Restart Kernel → Run All** bằng venv `D:\UIT\.venvs\ml-gpu`.
Mặc định chạy đủ **100 cấu hình LightGBM trên CPU**, toàn bộ `Base.csv`.
Không cần GPU. Các cấu hình có tới 10.000 cây nên lượt này có thể chạy lâu.

Notebook clone repo Feedzai, ghim commit, chạy lại các cell 0–13 của
`empirical_results.ipynb`, giữ các file gốc và bản diff trong proof.
Phạm vi chỉ có Base, không phải toàn bộ sáu biến thể của paper.

**Protocol gốc:** train tháng 0–5, test tháng 6–7. Cả ngưỡng FPR và
top-five được lấy trên test. Đây là tái lập thực nghiệm mô tả;
không dùng top-five này để tuyên bố cải thiện trên test độc lập.
Bộ test này cũng đã xuất hiện trong các thí nghiệm trước của nhóm.

Cách xử lý tương thích: dùng LightGBM hiện có (4.7.0), giữ Optuna 2.10.0;
đổi CSV/Base, sửa API Aequitas, `iteritems` → `items`, thêm seed cho shuffle
vốn không được ghim trong code gốc và thêm lưu evidence. Mọi sửa đổi được ghi rõ.

## 1. Project và nơi lưu cache

In [1]:
from pathlib import Path
import contextlib
import datetime
import difflib
import hashlib
import importlib.metadata
import io
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
import zipfile

PROJECT_DIR = None
here = Path.cwd().resolve()
ROOT = (
    Path(PROJECT_DIR).resolve()
    if PROJECT_DIR
    else next(
        (p for p in [here, *here.parents] if (p / "data/raw/Base.csv").is_file()),
        None,
    )
)
if ROOT is None:
    raise FileNotFoundError("Mở BAF_project hoặc điền PROJECT_DIR.")

for key, folder in [("PIP_CACHE_DIR", "pip-cache"), ("TEMP", "tmp"), ("TMP", "tmp")]:
    path = ROOT / "external" / "feedzai08_runtime" / folder
    path.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(path)


### Cấu hình cố định cho lượt tái lập

In [2]:
DATA_PATH = ROOT / "data/raw/Base.csv"
EXPECTED_ROWS = 1_000_000
EXPECTED_DATA_SHA256 = (
    "7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809"
)
REPO_URL = "https://github.com/feedzai/bank-account-fraud.git"
COMMIT = "3377cfd8fc20edcda3dd9dd14c10ed3dd276c6de"
REPO_DIR = ROOT / "external" / "feedzai_3377cfd8fc20"
RUNTIME_DIR = ROOT / "external" / "feedzai08_runtime" / "packages"
N_TRIALS = 100
N_JOBS = 10
SHUFFLE_SEED = 42
SOURCE_SHA256 = {
    "empirical_results.ipynb": (
        "8964df82c3ab1417aeaf39b15d1146d1" "3eddb9813f1d8f014f59387c79b920be"
    ),
    "random_search.py": (
        "f7e830468b6393e5cca90125e927640c" "ecec06234ca26e276885b40f90ec7e1d"
    ),
    ("lightgbm_hyperparameter_space.ya" "ml"): (
        "f991dadead1a99ac68d2a92b225610fc" "9412f6137333bf7815cda227f05594ed"
    ),
    "requirements.txt": (
        "f5805992a2bf8c373aedb4a6df192839" "0452d93f4fee3e4e3e1d4226aa01360e"
    ),
    "README.md": "043afc759f0818c790a659a33dcc0bfa11172567739614e74b3cad9d4aa16716",
    "LICENSE": "cd0438d90fde99cdc0db4971c9a85a9d53b7fe07923c318fdf74000bd110630b",
    "BAF_paper.pdf": "9a3369fa3c2a0cc9588889edc4c2e940cc7dd28e87a21a1aba632bf4512779b4",
    "datasheet.pdf": "367846738d63ed352183ba5e130809c8a313bd9219aa426c916ac2be709a81b0",
}


### Hàm đọc và lưu bằng chứng

In [3]:
def sha256_file(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )


if shutil.which("git") is None:
    raise RuntimeError("Cài Git for Windows rồi mở lại VS Code.")
if sha256_file(DATA_PATH) != EXPECTED_DATA_SHA256:
    raise ValueError("Base.csv khác dữ liệu đã dùng; kiểm tra trước khi train.")
print("Project:", ROOT)
print("Python:", sys.executable)


Project: D:\UIT\DS317\BAF_project
Python: d:\UIT\.venvs\ml-gpu\Scripts\python.exe


## 2. Kiểm tra phụ thuộc

Phụ thuộc cũ chỉ đặt trong `external/feedzai08_runtime/packages` trên ổ của project.

In [4]:
required = {
    "lightgbm": "lightgbm==4.7.0",
    "nbformat": "nbformat==5.11.1",
    "ipython": "ipython==9.17.1",
    "pyyaml": "PyYAML==6.0.3",
    "seaborn": "seaborn>=0.13.2",
}
missing = []
for name, requirement in required.items():
    try:
        importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

import nbformat

probe_lines = [
    "import sys",
    "import importlib.metadata as metadata",
    "sys.path.insert(0, sys.argv[1])",
    "import optuna",
    "from aequitas.group import Group",
    "assert optuna.__version__ == '2.10.0'",
    "assert metadata.version('aequitas') == '0.42.0'",
]
probe = "\n".join(probe_lines)
checked = subprocess.run(
    [sys.executable, "-c", probe, str(RUNTIME_DIR)], capture_output=True, text=True
)


In [5]:
if checked.returncode:
    legacy_packages = [
        "optuna==2.10.0",
        "aequitas==0.42.0",
        "alembic==1.20.1",
        "SQLAlchemy==2.1.6",
        "Mako==1.4.3",
        "MarkupSafe==3.0.4",
        "greenlet==3.5.6",
        "cmaes==0.13.1",
        "colorlog==6.12.0",
    ]
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            "--upgrade",
            "--only-binary=:all:",
            "--target",
            str(RUNTIME_DIR),
            *legacy_packages,
        ],
        check=True,
    )
    subprocess.run([sys.executable, "-c", probe, str(RUNTIME_DIR)], check=True)
print("Legacy sampler / Group: OK. Device: CPU.")


Legacy sampler / Group: OK. Device: CPU.


## 3. Clone repo và giữ nguyên tài liệu gốc

In [6]:
suffix = datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "results/runs" / ("feedzai_base_original_" + suffix)
EVIDENCE = RUN_DIR / "evidence"
EVIDENCE.mkdir(parents=True, exist_ok=False)
EXECUTED_PATH = RUN_DIR / "feedzai_base_executed.ipynb"


def git_command(arguments, binary=False):
    result = subprocess.run(
        ["git", "-C", str(REPO_DIR), *arguments],
        check=True,
        capture_output=True,
    )
    return result.stdout if binary else result.stdout.decode("utf-8").strip()


if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    git_command(["checkout", "--detach", COMMIT])
if git_command(["remote", "get-url", "origin"]) != REPO_URL:
    raise ValueError("Repo origin khác nguồn được ghim.")
if git_command(["rev-parse", "HEAD"]) != COMMIT:
    raise ValueError("Repo đang ở commit khác.")
if git_command(["status", "--porcelain"]):
    raise ValueError("Repo có thay đổi; giữ riêng bản gốc sạch.")


In [7]:
source_paths = [
    "notebooks/empirical_results.ipynb",
    "notebooks/random_search.py",
    "notebooks/lightgbm_hyperparameter_space.yaml",
    "notebooks/requirements.txt",
    "README.md",
    "LICENSE",
    "documents/BAF_paper.pdf",
    "documents/datasheet.pdf",
]
for relative in source_paths:
    data = git_command(["show", COMMIT + ":" + relative], binary=True)
    name = Path(relative).name
    if hashlib.sha256(data).hexdigest() != SOURCE_SHA256[name]:
        raise ValueError("Source hash mismatch: " + name)
    (EVIDENCE / name).write_bytes(data)

parameter_path = ROOT / "docs/feedzai08_parameters.json"
parameter_bytes = parameter_path.read_bytes()
if (
    hashlib.sha256(parameter_bytes).hexdigest()
    != "433f28a8381147c44f108346bc6449ad6a8fb5fcddeb37a9705d0448822c9057"
):
    raise ValueError("Bộ tham số tham chiếu bị sửa.")
(EVIDENCE / "expected_parameters.json").write_bytes(parameter_bytes)
shutil.copy2(
    EVIDENCE / "lightgbm_hyperparameter_space.yaml",
    RUN_DIR / "lightgbm_hyperparameter_space.yaml",
)
(EVIDENCE / "git.log").write_text(
    git_command(["log", "-5", "--format=fuller"]), encoding="utf-8"
)
write_json(
    EVIDENCE / "git_identity.json",
    {"remote": REPO_URL, "commit": COMMIT, "working_tree_clean": True},
)
print("Commit:", COMMIT)
print("Run:", RUN_DIR)


Commit: 3377cfd8fc20edcda3dd9dd14c10ed3dd276c6de
Run: D:\UIT\DS317\BAF_project\results\runs\feedzai_base_original_20261028_003248_73856e42


## 4. Hàm lưu trial, lớp tương thích và trình ghi output

Các file sau là evidence riêng của lượt chạy; không thêm module vào `src`.

In [8]:
CAPTURE_FILE = (EVIDENCE / "capture_results.py").as_posix()
RUNNER_FILE = (EVIDENCE / "execute_notebook.py").as_posix()
AUDIT_FILE = (EVIDENCE / "verify_results.py").as_posix()


### Lưu model, dự đoán và metric sau mỗi trial

In [9]:
%%writefile "{CAPTURE_FILE}"
import gzip
import importlib.metadata
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from aequitas.group import Group as OriginalGroup
from sklearn import metrics


class Group(OriginalGroup):
    """Map the notebook's keyword API to the pinned Aequitas 0.42 API."""

    def get_crosstabs(self, df, attr_cols=None, score_col=None, label_col=None):
        data = df.rename(columns={score_col: "score", label_col: "label_value"}).copy()
        for column in attr_cols:
            data[column] = data[column].astype(object)
        return super().get_crosstabs(data, attr_cols=attr_cols)


class QuietLightGBM:
    """Keep distinct warning messages without repeating per-tree notices."""

    def __init__(self):
        self.seen = set()

    def info(self, message):
        if "[Warning]" in message and message not in self.seen:
            self.seen.add(message)
            print(message, flush=True)

    def warning(self, message):
        self.info("[Warning] " + message)


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )


Writing D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/capture_results.py


In [10]:
%%writefile -a "{CAPTURE_FILE}"


def save_split_evidence():
    tables = []
    for name, frame in [("train", train_dfs["Base"]), ("test", test_dfs["Base"])]:
        tables.append(
            pd.DataFrame(
                {
                    "partition": name,
                    "row_index": frame.index,
                    "y_true": frame.fraud_bool.to_numpy(),
                    "month": frame.month.to_numpy(),
                    "customer_age": frame.customer_age.to_numpy(),
                }
            )
        )
    pd.concat(tables).to_csv("split_indices.csv.gz", index=False)
    categories = {
        column: sorted(train_dfs["Base"][column].unique().tolist())
        for column in categorical_features
    }
    write_json("preprocessing.json", {"label_encoder_classes": categories})
    Path("models").mkdir(exist_ok=True)
    Path("predictions").mkdir(exist_ok=True)
    Path("trials").mkdir(exist_ok=True)


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/capture_results.py


In [11]:
%%writefile -a "{CAPTURE_FILE}"


def capture_trial(number, seed, model, predictions, threshold, recall, results):
    stem = f"trial_{number:03d}"
    scores = np.asarray(predictions, dtype=np.float64)
    y = y_test.to_numpy(dtype=np.int64)
    binary = (scores > threshold).astype(np.int64)
    tn, fp, fn, tp = metrics.confusion_matrix(y, binary, labels=[0, 1]).ravel()
    np.savez_compressed(Path("predictions") / (stem + ".npz"), score=scores)
    with gzip.open(Path("models") / (stem + ".txt.gz"), "wt", encoding="utf-8") as f:
        f.write(model.booster_.model_to_string())
    row = {
        "trial": number,
        "sampling_seed": seed,
        "hyperparams": test_hyperparams.copy(),
        "fit_seconds": fit_seconds,
        "source_results": results.copy(),
        "source_threshold": float(threshold) if np.isfinite(threshold) else None,
        "source_threshold_is_posinf": bool(np.isposinf(threshold)),
        "source_roc_recall": float(recall),
        "binary_recall_strict_gt": float(metrics.recall_score(y, binary)),
        "binary_fpr_strict_gt": float(fp / (fp + tn)),
        "average_precision": float(metrics.average_precision_score(y, scores)),
        "roc_auc": float(metrics.roc_auc_score(y, scores)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fitted_trees": model.booster_.num_trees(),
    }
    write_json(Path("trials") / (stem + ".json"), row)
    print(
        f"FEEDZAI_TRIAL {number}/{N_TRIALS} "
        f"recall={recall:.6f} AP={row['average_precision']:.6f} "
        f"fit={fit_seconds:.1f}s",
        flush=True,
    )


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/capture_results.py


In [12]:
%%writefile -a "{CAPTURE_FILE}"


def finalize_results():
    rows = [
        json.loads(path.read_text(encoding="utf-8"))
        for path in sorted(Path("trials").glob("trial_*.json"))
    ]
    assert len(rows) == N_TRIALS
    write_json("all_trials.json", rows)
    rs_results.to_json(
        "source_results.json", orient="records", indent=2, double_precision=15
    )
    summary = pd.json_normalize(rows)
    summary.to_csv("trial_summary.csv", index=False)
    plot_results.to_csv("source_plot_results.csv", index=False)
    top = summary.sort_values("source_roc_recall", ascending=False, kind="stable").head(
        5
    )
    top.to_csv("top5_descriptive_test.csv", index=False)
    from matplotlib import pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_results["Recall"], plot_results["FPR Ratio"], alpha=0.5)
    ax.set(xlabel="Recall: source ROC point", ylabel="FPR ratio: min / max")
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig("base_recall_fairness.png", dpi=180)
    plt.close(fig)
    write_json(
        "runtime_versions.json",
        {
            "python": sys.version,
            "packages": {
                name: importlib.metadata.version(name)
                for name in [
                    "numpy",
                    "pandas",
                    "scipy",
                    "scikit-learn",
                    "lightgbm",
                    "optuna",
                    "aequitas",
                    "PyYAML",
                    "IPython",
                    "nbformat",
                ]
            },
            "device": "CPU",
        },
    )
    print("FEEDZAI_BASE_RESULTS_READY")
    print(top[["trial", "source_roc_recall", "average_precision", "fit_seconds"]])


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/capture_results.py


### Ghi output thực của từng cell vào notebook bằng chứng

In [13]:
%%writefile "{RUNNER_FILE}"
import contextlib
import io
import sys
import traceback
from pathlib import Path

import nbformat
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display
from IPython.utils.capture import capture_output


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
            stream.flush()
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


path = Path(sys.argv[1]).resolve()
notebook = nbformat.read(path, as_version=4)
shell = InteractiveShell.instance()
shell.user_ns["display"] = display
count = 0


Writing D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/execute_notebook.py


In [14]:
%%writefile -a "{RUNNER_FILE}"
for cell in notebook.cells:
    if cell.cell_type != "code":
        continue
    count += 1
    stdout = io.StringIO()
    stderr = io.StringIO()
    print("BEGIN_CELL", cell.metadata.get("author_cell", "evidence"), flush=True)
    with capture_output(stdout=False, stderr=False, display=True) as rich:
        with contextlib.redirect_stdout(Tee(sys.stdout, stdout)):
            with contextlib.redirect_stderr(Tee(sys.stderr, stderr)):
                result = shell.run_cell(cell.source, store_history=True)
    cell.execution_count = count
    cell.outputs = []
    for name, stream in [("stdout", stdout), ("stderr", stderr)]:
        if stream.getvalue():
            cell.outputs.append(
                nbformat.v4.new_output("stream", name=name, text=stream.getvalue())
            )
    for output in rich.outputs:
        cell.outputs.append(
            nbformat.v4.new_output(
                "display_data", data=output.data, metadata=output.metadata
            )
        )
    error = result.error_before_exec or result.error_in_exec
    if error:
        cell.outputs.append(
            nbformat.v4.new_output(
                "error",
                ename=type(error).__name__,
                evalue=str(error),
                traceback=traceback.format_exception(
                    type(error), error, error.__traceback__
                ),
            )
        )
    nbformat.write(notebook, path)
    if error:
        raise RuntimeError("Author notebook execution failed") from error
    print("END_CELL", cell.metadata.get("author_cell", "evidence"), flush=True)
print("EXECUTED_NOTEBOOK_SAVED", path, flush=True)


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/execute_notebook.py


### Kiểm chứng độc lập từ file dự đoán

In [15]:
%%writefile "{AUDIT_FILE}"
from pathlib import Path
import gzip
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn import metrics

PROOF_DIR = None
here = Path.cwd().resolve()
root = (
    Path(PROOF_DIR).resolve()
    if PROOF_DIR
    else next(
        (p for p in [here, *here.parents] if (p / "author_run.json").is_file()),
        None,
    )
)
if root is None:
    raise FileNotFoundError("Open the extracted proof folder or set PROOF_DIR")


def read_json(name):
    return json.loads((root / name).read_text(encoding="utf-8"))


def same(left, right):
    np.testing.assert_allclose(left, right, rtol=1e-10, atol=1e-12)


Writing D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/verify_results.py


In [16]:
%%writefile -a "{AUDIT_FILE}"


PINNED_SOURCE_SHA256 = {
    "empirical_results.ipynb": (
        "8964df82c3ab1417aeaf39b15d1146d1" "3eddb9813f1d8f014f59387c79b920be"
    ),
    "random_search.py": (
        "f7e830468b6393e5cca90125e927640c" "ecec06234ca26e276885b40f90ec7e1d"
    ),
    ("lightgbm_hyperparameter_space.ya" "ml"): (
        "f991dadead1a99ac68d2a92b225610fc" "9412f6137333bf7815cda227f05594ed"
    ),
    "requirements.txt": (
        "f5805992a2bf8c373aedb4a6df192839" "0452d93f4fee3e4e3e1d4226aa01360e"
    ),
    "README.md": "043afc759f0818c790a659a33dcc0bfa11172567739614e74b3cad9d4aa16716",
    "LICENSE": "cd0438d90fde99cdc0db4971c9a85a9d53b7fe07923c318fdf74000bd110630b",
    "BAF_paper.pdf": "9a3369fa3c2a0cc9588889edc4c2e940cc7dd28e87a21a1aba632bf4512779b4",
    "datasheet.pdf": "367846738d63ed352183ba5e130809c8a313bd9219aa426c916ac2be709a81b0",
}

info = read_json("author_run.json")
assert info["source_sha256"] == PINNED_SOURCE_SHA256
assert info["state"] == "completed"
assert info["commit"] == "3377cfd8fc20edcda3dd9dd14c10ed3dd276c6de"
assert info["n_trials"] > 0
for name, expected in info["source_sha256"].items():
    actual = hashlib.sha256((root / "evidence" / name).read_bytes()).hexdigest()
    assert actual == expected, name
original = read_json("evidence/empirical_results.ipynb")
executed = read_json("feedzai_base_executed.ipynb")
mapping = read_json("evidence/cell_mapping.json")
assert [row["author_cell"] for row in mapping] == list(range(14))
for row in mapping:
    index = row["author_cell"]
    source = "".join(original["cells"][index]["source"])
    cell = next(
        c
        for c in executed["cells"]
        if c.get("metadata", {}).get("author_cell") == index
    )
    effective = "".join(cell["source"])
    assert hashlib.sha256(source.encode()).hexdigest() == row["original_sha256"]
    assert hashlib.sha256(effective.encode()).hexdigest() == row["executed_sha256"]
    assert row["modified"] == (source != effective)
    assert cell["execution_count"] is not None


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/verify_results.py


In [17]:
%%writefile -a "{AUDIT_FILE}"
assert not [
    o
    for c in executed["cells"]
    for o in c.get("outputs", [])
    if o["output_type"] == "error"
]

splits = pd.read_csv(root / "split_indices.csv.gz")
assert len(splits) == info["data_rows"]
assert np.array_equal(np.sort(splits.row_index), np.arange(len(splits)))
assert set(splits.partition) == {"train", "test"}
assert set(splits.y_true) == {0, 1}
assert (splits.loc[splits.partition == "train", "month"] < 6).all()
assert (splits.loc[splits.partition == "test", "month"] >= 6).all()
test = splits[splits.partition == "test"]
for frame in [splits[splits.partition == "train"], test]:
    original_order = frame.sort_values("row_index").row_index
    if frame.partition.iloc[0] == "train":
        rng = np.random.RandomState(info["shuffle_seed"])
    expected = rng.permutation(len(original_order))
    assert np.array_equal(frame.row_index, original_order.to_numpy()[expected])
y = test.y_true.to_numpy(dtype=np.int64)
ages = test.customer_age.to_numpy()
records = read_json("all_trials.json")
expected_params = read_json("evidence/expected_parameters.json")
assert len(records) == info["n_trials"] <= 100
seeds = np.random.RandomState(42).choice(
    1_000_000, size=info["n_trials"], replace=False
)
assert seeds.tolist() == [r["sampling_seed"] for r in records]
source_results = read_json("source_results.json")
plot = pd.read_csv(root / "source_plot_results.csv")


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/verify_results.py


In [18]:
%%writefile -a "{AUDIT_FILE}"

for number, row in enumerate(records, 1):
    stem = f"trial_{number:03d}"
    assert row["trial"] == number
    assert row == read_json(f"trials/{stem}.json")
    expected = expected_params["trials"][number - 1]
    assert row["sampling_seed"] == expected["seed"]
    assert row["hyperparams"] == expected["hyperparams"]
    assert row["fit_seconds"] > 0
    with np.load(root / "predictions" / (stem + ".npz"), allow_pickle=False) as saved:
        scores = saved["score"]
    assert scores.dtype == np.float64 and scores.shape == y.shape
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
    fpr, tpr, thresholds = metrics.roc_curve(y, scores)
    selected_fpr = max(fpr[fpr < 0.05])
    threshold = np.min(thresholds[fpr == selected_fpr])
    recall = np.max(tpr[fpr == selected_fpr])
    if np.isposinf(threshold):
        assert row["source_threshold"] is None
        assert row["source_threshold_is_posinf"] is True
    else:
        same(threshold, row["source_threshold"])
        assert row["source_threshold_is_posinf"] is False
    same(recall, row["source_roc_recall"])
    same(recall, row["source_results"]["recall"])
    binary = (scores > threshold).astype(np.int64)
    tn, fp, fn, tp = metrics.confusion_matrix(y, binary, labels=[0, 1]).ravel()
    assert [int(tn), int(fp), int(fn), int(tp)] == [
        row[k] for k in ["tn", "fp", "fn", "tp"]
    ]
    same(tp / (tp + fn), row["binary_recall_strict_gt"])
    same(fp / (fp + tn), row["binary_fpr_strict_gt"])
    same(metrics.average_precision_score(y, scores), row["average_precision"])
    same(metrics.roc_auc_score(y, scores), row["roc_auc"])
    group_fprs = []
    for name, mask in [("Older", ages >= 50), ("Younger", ages < 50)]:
        a, b, c, d = metrics.confusion_matrix(
            y[mask], binary[mask], labels=[0, 1]
        ).ravel()
        assert a + b > 0 and c + d > 0
        group_fprs.append(b / (a + b))
        same(b / (a + b), row["source_results"]["fpr " + name])
        same(d / (c + d), row["source_results"]["recall " + name])
    for key, value in row["source_results"].items():
        same(value, source_results[number - 1]["Base"][key])
    ratio = min(group_fprs) / max(group_fprs) if max(group_fprs) > 0 else None
    if ratio is not None:
        same(ratio, plot.iloc[number - 1]["FPR Ratio"])
    else:
        assert pd.isna(plot.iloc[number - 1]["FPR Ratio"])
    model_path = root / "models" / (stem + ".txt.gz")
    with gzip.open(model_path, "rt", encoding="utf-8") as f:
        model_text = f.read()
    assert model_text.startswith("tree\n")
    assert model_text.count("\nTree=") == row["fitted_trees"]


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/verify_results.py


In [19]:
%%writefile -a "{AUDIT_FILE}"

hash_path = root / "sha256.json"
if hash_path.is_file():
    hashes = read_json("sha256.json")
    for name, expected in hashes.items():
        actual = hashlib.sha256((root / name).read_bytes()).hexdigest()
        assert actual == expected, name
    print("Verified SHA-256 entries:", len(hashes))
print("FEEDZAI_BASE_PROOF_VERIFIED", len(records), "trials")
print("Scope:", info["scope"])
print("Test rows:", len(test), "Fraud:", int(y.sum()))
print("Threshold is selected on test; top-five is descriptive test selection.")


Appending to D:/UIT/DS317/BAF_project/results/runs/feedzai_base_original_20261028_003248_73856e42/evidence/verify_results.py


## 5. Chuẩn bị notebook chạy các cell tác giả

In [20]:
original = nbformat.read(EVIDENCE / "empirical_results.ipynb", as_version=4)
capture_source = Path(CAPTURE_FILE).read_text(encoding="utf-8")
audit_source = Path(AUDIT_FILE).read_text(encoding="utf-8")
setup = [
    "import sys",
    "import os",
    "import json",
    "import time",
    "import importlib.metadata",
    "from pathlib import Path",
    "sys.dont_write_bytecode = True",
    "sys.path.insert(0, " + repr(str(RUNTIME_DIR)) + ")",
    "sys.path.insert(0, " + repr(str(EVIDENCE)) + ")",
    "import numpy as np",
    "np.random.seed(" + repr(SHUFFLE_SEED) + ")",
    "DATA_PATH = " + repr(str(DATA_PATH)),
    "N_TRIALS = " + repr(N_TRIALS),
    "N_JOBS = " + repr(N_JOBS),
    "EXPECTED_ROWS = " + repr(EXPECTED_ROWS),
    "exec(Path(" + repr(CAPTURE_FILE) + ").read_text(encoding='utf-8'), globals())",
]
execution = nbformat.v4.new_notebook(
    cells=[
        nbformat.v4.new_markdown_cell(
            "# Feedzai Base: actual author-cell outputs\n\n"
            "100-trial CPU replay with documented compatibility changes."
        ),
        nbformat.v4.new_code_cell("\n".join(setup)),
    ],
    metadata={"kernelspec": {"name": "python3", "display_name": "Python 3"}},
)
mapping = []
changes = []


In [21]:
def adapted_source(index, source):
    effective = source
    if index == 1:
        effective = effective.replace(
            "from aequitas.group import Group",
            "# Group is provided by the documented Aequitas 0.42 adapter.",
        )
        effective += "\nlgbm.register_logger(QuietLightGBM())\n"
    elif index == 5:
        effective = 'datasets_paths = {"Base": str(DATA_PATH)}\n'
    elif index == 6:
        effective = source.replace("pd.read_parquet(path)", "pd.read_csv(path)")
        effective += (
            "\nassert len(datasets['Base']) == EXPECTED_ROWS\n"
            "assert set(datasets['Base'].fraud_bool.unique()) == {0, 1}\n"
        )
    elif index == 8:
        effective += "\nsave_split_evidence()\n"
    elif index == 10:
        effective = effective.replace("n_trials=100", "n_trials=N_TRIALS")
        effective = effective.replace("n_jobs=10", "n_jobs=N_JOBS")
        marker = (
            "        model.fit(X_train, y_tra"
            "in, categorical_feature=categori"
            "cal_features)"
        )
        replacement = (
            '        print("FIT_TRIAL", len(runs.get(dataset_name, [])) + 1, '
            "test_hyperparams, flush=True)\n"
            "        fit_started = time.perf_counter()\n" + marker + "\n"
            "        fit_seconds = time.perf_counter() - fit_started"
        )
        assert effective.count(marker) == 1
        effective = effective.replace(marker, replacement)
        marker = "        runs[dataset_name] = prev_runs"
        effective = effective.replace(
            marker,
            marker + "\n        capture_trial(len(prev_runs), int(seed), model, "
            "predictions, threshold, recall, results)\n",
        )
    elif index == 12:
        effective = source.replace(".iteritems()", ".items()")
    return effective


In [22]:
for index in range(14):
    source = original.cells[index].source
    effective = adapted_source(index, source)
    mapping.append(
        {
            "author_cell": index,
            "original_sha256": hashlib.sha256(source.encode()).hexdigest(),
            "executed_sha256": hashlib.sha256(effective.encode()).hexdigest(),
            "modified": effective != source,
        }
    )
    if effective != source:
        changes.extend(
            difflib.unified_diff(
                source.splitlines(True),
                effective.splitlines(True),
                fromfile=f"original/cell_{index}",
                tofile=f"executed/cell_{index}",
            )
        )
    execution.cells.append(nbformat.v4.new_markdown_cell(f"## Author cell {index}"))
    cell = nbformat.v4.new_code_cell(effective)
    cell.metadata["author_cell"] = index
    execution.cells.append(cell)
execution.cells.append(nbformat.v4.new_code_cell("finalize_results()"))
nbformat.write(execution, EXECUTED_PATH)
write_json(EVIDENCE / "cell_mapping.json", mapping)
(EVIDENCE / "source_changes.diff").write_text("".join(changes), encoding="utf-8")


4120

### Phạm vi và môi trường của lượt chạy

In [23]:
run_info = {
    "state": "prepared",
    "repo": REPO_URL,
    "commit": COMMIT,
    "source_sha256": SOURCE_SHA256,
    "data_sha256": EXPECTED_DATA_SHA256,
    "data_rows": EXPECTED_ROWS,
    "n_trials": N_TRIALS,
    "n_jobs": N_JOBS,
    "shuffle_seed": SHUFFLE_SEED,
    "scope": "Base only; original author cells 0-13; CPU; temporal train<6/test>=6",
    "full_base_scope": N_TRIALS == 100 and EXPECTED_ROWS == 1_000_000,
    "sampling": "Original random_search.py with Optuna 2.10.0 and YAML",
    "threshold": "Original test ROC: largest FPR strictly below 0.05",
    "binary_rule": "Original predictions > threshold; ROC uses >= threshold",
    "selection": "Top-five ranked on test for description; not held-out selection",
    "environment_note": "Modern LightGBM; not the original Python3.7/LGBM3.1.1 stack",
    "prior_test_exposure": "Months6/7 have appeared in earlier group experiments",
    "shuffle_change": "Author shuffle unseeded; seeded NumPy42 before author cells",
    "compatibility": "CSV/Base scope, Aequitas column API, pandas items, observer",
    "python_executable": sys.executable,
}
write_json(RUN_DIR / "author_run.json", run_info)
print("Prepared:", EXECUTED_PATH)
print("Trials:", N_TRIALS, "CPU threads:", N_JOBS)


Prepared: D:\UIT\DS317\BAF_project\results\runs\feedzai_base_original_20261028_003248_73856e42\feedzai_base_executed.ipynb
Trials: 100 CPU threads: 10


## 6. Chạy 100 trial

Mỗi trial lưu model và dự đoán ngay sau khi hoàn tất. Ngắt chạy vẫn giữ evidence của các trial đã xong; Run All tạo run mới.

In [24]:
if run_info["state"] != "prepared":
    raise RuntimeError("Restart Kernel + Run All để tạo run mới.")
child_env = os.environ.copy()
child_env["PYTHONUNBUFFERED"] = "1"
child_env["PYTHONIOENCODING"] = "utf-8"
child_env["PYTHONDONTWRITEBYTECODE"] = "1"
child_env["MPLBACKEND"] = "Agg"
run_info["state"] = "running"
run_info["execution_engine"] = "Fresh Python process with IPython InteractiveShell"
run_info["started"] = datetime.datetime.now().astimezone().isoformat()
write_json(RUN_DIR / "author_run.json", run_info)
process = None
started = time.perf_counter()


In [25]:
try:
    with (RUN_DIR / "training.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            [sys.executable, "-u", str(Path(RUNNER_FILE)), str(EXECUTED_PATH)],
            cwd=RUN_DIR,
            env=child_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(
            (
                f"Author execution failed ({return_code}); "
                "xem training.log và notebook partial."
            )
        )
    run_info["state"] = "completed"
except BaseException as error:
    run_info["state"] = "failed_or_interrupted"
    run_info["error"] = repr(error)
    raise
finally:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    run_info["elapsed_seconds"] = time.perf_counter() - started
    run_info["finished"] = datetime.datetime.now().astimezone().isoformat()
    write_json(RUN_DIR / "author_run.json", run_info)

print("Feedzai Base completed (minutes):", run_info["elapsed_seconds"] / 60)


BEGIN_CELL evidence
END_CELL evidence
BEGIN_CELL 0
END_CELL 0
BEGIN_CELL 1
END_CELL 1
BEGIN_CELL 2
END_CELL 2
BEGIN_CELL 3
Out[5]: 
{'LightGBM': {'classpath': 'lightgbm.LGBMClassifier',
  'kwargs': {'n_estimators': {'type': 'int',
    'range': [20, 10000],
    'log': True},
   'max_depth': {'type': 'int', 'range': [3, 30]},
   'learning_rate': {'type': 'float', 'range': [0.02, 0.1], 'log': True},
   'num_leaves': {'type': 'int', 'range': [10, 100], 'log': True},
   'boosting_type': ['gbdt', 'goss'],
   'min_data_in_leaf': {'type': 'int', 'range': [5, 200], 'log': True},
   'max_bin': {'type': 'int', 'range': [100, 500]},
   'enable_bundle': [True, False]}}}
END_CELL 3
BEGIN_CELL 4
Out[6]: 
{'classpath': 'lightgbm.LGBMClassifier',
 'n_estimators': 263,
 'max_depth': 23,
 'learning_rate': 0.020003681922217444,
 'num_leaves': 19,
 'boosting_type': 'gbdt',
 'min_data_in_leaf': 9,
 'max_bin': 238,
 'enable_bundle': False}
END_CELL 4
BEGIN_CELL 5
END_CELL 5
BEGIN_CELL 6
END_CELL 6
BEGIN_CELL

## 7. Kiểm chứng và xuất proof một lượt

Cell này tính lại metric từ dự đoán đã lưu; không train thêm và không phụ thuộc Ctrl+S.

In [26]:
audit_text = audit_source.replace(
    "PROOF_DIR = None", "PROOF_DIR = " + repr(str(RUN_DIR)), 1
)
audit_stdout = io.StringIO()
with contextlib.redirect_stdout(audit_stdout):
    exec(compile(audit_text, "<Feedzai Base audit>", "exec"), {"__name__": "__audit__"})
print(audit_stdout.getvalue())
verify = nbformat.v4.new_notebook(
    cells=[
        nbformat.v4.new_markdown_cell(
            "# Verify Feedzai Base proof\n\n"
            "Extract, open this directory, Run All. No raw CSV/GPU/training needed."
        ),
        nbformat.v4.new_code_cell(
            audit_source,
            execution_count=1,
            outputs=[
                nbformat.v4.new_output(
                    "stream", name="stdout", text=audit_stdout.getvalue()
                )
            ],
        ),
    ]
)
nbformat.write(verify, RUN_DIR / "verify.ipynb")
write_json(
    RUN_DIR / "export_status.json",
    {"state": "completed", "training_performed_by_export": False},
)
hashes = {
    p.relative_to(RUN_DIR).as_posix(): sha256_file(p)
    for p in RUN_DIR.rglob("*")
    if p.is_file() and p.name != "sha256.json"
}
write_json(RUN_DIR / "sha256.json", hashes)


FEEDZAI_BASE_PROOF_VERIFIED 100 trials
Scope: Base only; original author cells 0-13; CPU; temporal train<6/test>=6
Test rows: 205011 Fraud: 2878
Threshold is selected on test; top-five is descriptive test selection.



In [27]:
PROOF_ROOT = ROOT / "results/proofs"

PROOF_ROOT.mkdir(parents=True, exist_ok=True)

archive = PROOF_ROOT / (RUN_DIR.name + "_proof_" + uuid.uuid4().hex[:8] + ".zip")

with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
    for path in RUN_DIR.rglob("*"):
        if path.is_file():
            bundle.write(path, Path(RUN_DIR.name) / path.relative_to(RUN_DIR))

print("Proof ZIP:", archive)

print("Original training notebook:", EXECUTED_PATH)


Proof ZIP: D:\UIT\DS317\BAF_project\results\proofs\feedzai_base_original_20261028_003248_73856e42_proof_cdd2794b.zip
Original training notebook: D:\UIT\DS317\BAF_project\results\runs\feedzai_base_original_20261028_003248_73856e42\feedzai_base_executed.ipynb


Khi hoàn tất, giữ ZIP proof và lưu notebook 08 để tiện đọc.
Proof có notebook chạy các cell gốc, log, model và dự đoán của từng trial,
paper/datasheet gốc, source diff, metric và `verify.ipynb`.
`top5_descriptive_test.csv` chỉ mô tả lượt tái lập, không phải cấu hình đã chọn
hợp lệ trên validation cho thí nghiệm cải thiện tiếp theo.